# Recover the existing Badenoch final 4K edit
Default **copy_exact** copies the already finished MP4 from this repository straight into Google Drive. No rerender, asset generation or external upload token is needed.

To rebuild the identical editorial timeline from your original master, change MODE to **render_original**. That runs the preserved final script, original screenshot/portrait pixels, frozen normalized narration, fonts and exact frame cuts. Different FFmpeg/libx264 builds can produce different encoded bytes. Only copy_exact guarantees the original MP4 checksum.

Run all cells in a standard Google Colab CPU runtime. Allow Drive mounting when prompted. Existing different final files are never overwritten. See final_recovery/README.md for source placement and historical review concerns.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
PROJECT = Path('/content/drive/MyDrive/Badenoch_VHS_Project')
MODE = 'copy_exact'  # or 'render_original'
SOURCE_OVERRIDE = None  # optional full path to the original master in mounted Drive


In [ ]:
import subprocess, sys
REPO = Path('/content/badenoch-final-recovery')
URL = 'https://github.com/SymplyFlorence/badenoch-vhs-video-editor.git'
REF = 'badenoch-final-recovery-v1'
if not REPO.exists():
    subprocess.run(['git','clone','--branch',REF,'--single-branch',URL,str(REPO)],check=True)
else:
    actual = subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
    expected = subprocess.check_output(['git','-C',str(REPO),'rev-parse',REF],text=True).strip()
    if actual != expected: raise RuntimeError('Existing Colab checkout is on another revision; use a fresh runtime.')
subprocess.run(['apt-get','-qq','update'],check=True)
subprocess.run(['apt-get','-qq','install','-y','ffmpeg'],check=True)
subprocess.run([sys.executable,'-m','pip','install','--quiet','Pillow==12.3.0'],check=True)


Preflight verifies SHA-256 for every preserved asset. Rebuild mode also requires the exact 2,009,538,306-byte original master; it will reject the compressed preview or a changed source. No missing asset is replaced.

In [ ]:
command = [sys.executable,str(REPO/'final_recovery/recover.py'),'--project',str(PROJECT),'--mode',MODE]
if SOURCE_OVERRIDE: command += ['--source',SOURCE_OVERRIDE]
subprocess.run(command+['--check-only'],check=True)


In [ ]:
subprocess.run(command,check=True)
output = PROJECT/'Final/Badenoch_documentary_FINAL_4K.mp4'
if not output.is_file(): raise RuntimeError('Final MP4 was not written to mounted Drive')
print('Saved directly in Google Drive:', output, output.stat().st_size, 'bytes')
